# Lesson 1: Introduction to RAG

## What is Retrieval-Augmented Generation (RAG)?

Retrieval-Augmented Generation (RAG) is an architectural pattern that combines the strengths of **retrieval systems** and **generative AI** to create more accurate, reliable, and context-aware AI systems.

## Why RAG?

Traditional LLMs have limitations:

1. **Hallucinations** - May invent facts that sound plausible but aren't true
2. **Knowledge cutoff** - Don't know information after their training date
3. **No access to private data** - Cannot use your organization's documents
4. **Generic responses** - Lack specific context for specialized queries

RAG solves these problems by:
- Retrieving relevant documents before generating responses
- Grounding answers in factual evidence
- Enabling access to up-to-date and private information
- Reducing hallucinations through evidence-based generation

## RAG vs Traditional LLMs\n\nLet's see the difference in action. Below, we'll make queries to demonstrate how RAG provides more accurate, evidence-based responses.

### Setup: Import Required Libraries

We'll use LangChain for building RAG pipelines. Install dependencies if needed:

In [ ]:
# Install dependencies if not already installed
import sys
packages = ['langchain', 'langchain-community', 'langchain-core', 'langchain-text-splitters', 'python-dotenv']
missing = [p for p in packages if not __import__('pkg_resources').working_set.by_key.get(p)]
if missing:
    __import__('sys').exit(__import__('subprocess').run(['pip', 'install'] + missing + ['-q']).returncode)
else:
    print('All required packages are installed.')

### Step 1: Import LangChain Components

We'll need these core components:

In [ ]:
# Import LangChain components
from langchain_community.llms import Ollama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

print('LangChain components imported successfully!')

### Step 2: Configure Your LLM and Embedding Models

You need to set up your API credentials. Check your `.env` file for these variables:

- `BASE_URL` - Your LLM endpoint (e.g., `http://localhost:11434` for Ollama)
- `LLM_MODEL` - Name of your LLM (e.g., `llama3`, `mistral`)  
- `EMBEDDING_MODEL` - Type of your embedding model (e.g., `nomic-embed-text`)

Let's test the connection:

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Configuration
BASE_URL = os.getenv('BASE_URL', 'http://localhost:11434')
LLM_MODEL = os.getenv('LLM_MODEL', 'llama3')
EMBEDDING_MODEL = os.getenv('EMBEDDING_MODEL', 'nomic-embed-text')

print(f'Using LLM: {BASE_URL}/{LLM_MODEL}')
print(f'Using Embeddings: {EMBEDDING_MODEL}')

# Initialize LLM with default timeout
llm = Ollama(
    base_url=BASE_URL,
    model=LLM_MODEL,
    timeout=300
)

# Initialize embedder with default timeout  
embeddings = OllamaEmbeddings(
    base_url=BASE_URL,
    model=EMBEDDING_MODEL,
    timeout=300
)

# Test the LLM
test_prompt = "Explain what RAG is in one sentence."
response = llm.invoke(test_prompt)
print(f'Test response: {response}')
print('LLM connection successful!')

### Traditional LLM Query

Let's see what a traditional LLM (without retrieval) would say about a specific topic:

In [ ]:
# Traditional LLM query without retrieval
question = "What are the key features of the Philippine Civil Code of 1949?"

response = llm.invoke(question)

print("=" * 50)
print("QUESTION:", question)
print("-" * 60)
print("TRADITIONAL LLM RESPONSE:")
print(response)

### Sample Text for RAG Demonstration

For this demonstration, let's create a sample document to demonstrate the RAG pipeline:

In [ ]:
# Sample legal text for demonstration
sample_text = """
The Philippine Civil Code of 1949, enacted under Commonwealth Act No. 625,
replaces the Civil Code of 1889 (derived from Spanish law) as the fundamental
source of private law in the Philippines. It governs obligations, contracts,
family relations, succession, and other civil matters.

Key Features:
1. Obligations and Contracts - Governs civil obligations arising from
   law, contracts, quasi-contracts, crimes, and quasi-delicts.
2. Property Rights - Establishes rights over things and property ownership.
3. Succession - Regulates inheritance and succession of property.
4. Family Law - Governs marriage, family relations, and domestic matters.
5. Particular Laws - Includes provisions on judicial proceedings,
   public property, and legal capacity.

The Code consists of 3000 articles and forms the foundation of Philippine civil law,
though it has been amended several times by subsequent laws. Article 11 defines
obligations as juridical ties established by which one person is bound to render
to another a prestation consisting in giving, doing, or not doing.
"""

print("Sample text loaded successfully!")
print("=" * 60)
print(sample_text[:250], "...")
print("=" * 60)

### Text Chunking

RAG systems work best with smaller text chunks retrieved individually. Let's split our document:

In [ ]:
# Set up text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)

# Split the text (clean newlines first)
cleaned_text = '\n\n'.join([line for line in sample_text.strip().split('\n') if line.strip()])
chunks = text_splitter.split_text(cleaned_text)

print(f"Original text length: {len(sample_text)} characters")
print(f"Number of chunks created: {len(chunks)}")
print("\nFirst chunk sample (first 300 characters):")
print("-" * 60)
print(chunks[0][:300])
print("...")
print("-" * 60)

print("\nText chunking successful!")

### Creating Vector Store and Retriever

Now we'll convert text chunks into vectors and store them in a vector database (FAISS):

In [ ]:
# Create the embeddings model
embeddings = OllamaEmbeddings(
    base_url=BASE_URL,
    model=EMBEDDING_MODEL,
    timeout=300
)

# Create FAISS vector store
vectorstore = FAISS.from_texts(
    texts=chunks,
    embedding=embeddings
)

print(f"Created vector store with {len(chunks)} chunks")
print(f"Vector dimension: {vectorstore.embeddings.embed_dim}")

# Create retriever - retrieve top 2 most similar chunks
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 2}
)

print("Retriever ready!")

### Running a RAG Query

We'll set up a prompt template and query the system with the Philippine Civil Code question. The retriever will find relevant chunks, which we'll pass to the LLM along with the question:

In [ ]:
# RAG prompt template
rag_prompt_template = """\n\nGiven the following context, answer the question. If you don't know, say 'I don't have enough information'.\n\nContext: {context}\n\nQuestion: {question}\n\nAnswer:\n"""

# Create prompt
prompt = PromptTemplate.from_template(rag_prompt_template)

# RAG query
question = "What are the main features of the Philippine Civil Code?"

# Retrieve relevant chunks
docs = retriever.get_relevant_documents(question)

print(f"Retrieved {len(docs)} relevant chunks:")
print("=" * 60)
context_text = ""
for i, doc in enumerate(docs):
    print(f"\nChunk {i+1}:")
    print("-" * 40)
    print(doc.page_content)
    print("-" * 40)
    context_text += doc.page_content + "\n"

# Query LLM with context
context_text_truncated = context_text[:1000]
rag_response = llm.invoke(prompt.invoke({'context': context_text_truncated, 'question': question}))

print("\n" + "=" * 60)
print("RAG RESPONSE:")
print("-" * 60)
print(rag_response)

### Comparison: Traditional LLM vs RAG

Let's compare the two responses side-by-side with the retrieved context.

In [ ]:
# Re-run traditional query for comparison
traditional_response = llm.invoke(question)

# Re-run RAG for comparison
docs = retriever.get_relevant_documents(question)
context_text = '\n\n'.join([doc.page_content for doc in docs])
context_text = context_text[:1000]
rag_prompt = PromptTemplate.from_template(rag_prompt_template)
rag_response = llm.invoke(rag_prompt.invoke({'context': context_text, 'question': question}))

print("COMPARISON")
print("=" * 70)

print("\n[Traditional LLM (no context):]")
print("-" * 70)
print(traditional_response)
print("-" * 70)

print("\n[With Document Context:]\n\nRetrieved context:")
print("-" * 50)
for doc in docs:
    print(f"Chunk 1: {doc.page_content[:200]}...")
print("-" * 50)
print("\nRAG Response:")
print("-" * 70)
print(rag_response)
print("-" * 70)

print("\nKey Differences:")
print("- RAG uses actual document information (Grounded in facts)")
print("- Traditional LLM relies only on training data (May hallucinate)")
print("- RAG can be updated with new documents (No knowledge cutoff)")
print("- RAG shows evidence by returning source documents")

## Summary

### What We Learned:

1. **RAG Architecture** - Combines retrieval with generation for more accurate answers
2. **Vector Embeddings** - Converting text to numerical vectors for similarity search
3. **Chunking** - Breaking documents into manageable pieces for retrieval
4. **Query Process** - Retrieve relevant chunks → Feed to LLM → Generate answer

### Key Benefits of RAG:

- **Reduced Hallucinations** - Answers are grounded in retrieved documents
- **Up-to-date Information** - Can use the latest documents
- **Private Data Access** - Can retrieve from your internal documents
- **Citation & Attribution** - Can show which documents informed the answer
- **Domain Specialization** - Tailored to specific domains/legal/medical/etc.

## Next Lesson Preview

In Lesson 2, we'll explore:
- Document stores and vector databases in depth
- Different embedding models and use cases
- Advanced retrieval strategies
- Context management for better answers